# Day 020 Project Solution — Second Brain Chatbot

A complete RAG chatbot over a personal knowledge base, with scripted Q&A and a mini eval run.

In [ ]:
import ollama
import chromadb


## RAG Pipeline

In [ ]:
def chunk_text(text: str, chunk_size: int = 300, overlap: int = 50) -> list[str]:
    words = text.split()
    step = chunk_size - overlap
    if step <= 0:
        step = 1
    chunks = []
    for i in range(0, len(words), step):
        chunk = " ".join(words[i : i + chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks


def embed_text(text: str, model: str = "nomic-embed-text") -> list[float]:
    return ollama.embeddings(model=model, prompt=text)["embedding"]


def build_index(
    docs: dict,
    collection_name: str = "second_brain",
    chunk_size: int = 300,
    overlap: int = 50,
):
    client = chromadb.Client()
    try:
        client.delete_collection(collection_name)
    except Exception:
        pass
    collection = client.create_collection(collection_name)
    for source, text in docs.items():
        chunks = chunk_text(text, chunk_size, overlap)
        ids, embeddings, documents, metadatas = [], [], [], []
        for i, chunk in enumerate(chunks):
            ids.append(f"{source}__{i}")
            embeddings.append(embed_text(chunk))
            documents.append(chunk)
            metadatas.append({"source": source, "chunk_index": i})
        if ids:
            collection.add(
                ids=ids, embeddings=embeddings,
                documents=documents, metadatas=metadatas,
            )
    return collection


def retrieve(query: str, collection, n_results: int = 3) -> list[dict]:
    if collection.count() == 0:
        return []
    emb = embed_text(query)
    actual_n = min(n_results, collection.count())
    results = collection.query(query_embeddings=[emb], n_results=actual_n)
    return [
        {"text": doc, "source": meta["source"], "distance": dist}
        for doc, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        )
    ]


def build_cited_prompt(question: str, chunks: list[dict]) -> str:
    context = "\n\n".join(
        f"[{i+1}] Source: {c['source']}\n{c['text']}"
        for i, c in enumerate(chunks)
    )
    return f"Context:\n{context}\n\nQuestion: {question}"


RAG_SYSTEM_PROMPT = (
    "You are a helpful assistant. Answer questions using ONLY the "
    "provided context. Cite the source numbers you used (e.g. [1], [2]). "
    "If the answer is not in the context, say 'I don't know.'"
)


def rag_answer(
    question: str,
    collection,
    model: str = "llama3.2",
    n_results: int = 3,
) -> dict:
    chunks = retrieve(question, collection, n_results)
    if not chunks:
        return {"answer": "I don't know.", "sources": []}
    prompt = build_cited_prompt(question, chunks)
    response = ollama.chat(model=model, messages=[
        {"role": "system", "content": RAG_SYSTEM_PROMPT},
        {"role": "user",   "content": prompt},
    ])
    answer  = response["message"]["content"]
    sources = list(dict.fromkeys(c["source"] for c in chunks))
    return {"answer": answer, "sources": sources}

## SecondBrainChatbot

In [ ]:
class SecondBrainChatbot:
    def __init__(self, docs: dict, model: str = 'llama3.2',
                 chunk_size: int = 300, overlap: int = 50):
        self.model = model
        self.collection = build_index(docs, chunk_size=chunk_size, overlap=overlap)
        self._history: list[dict] = []

    def ask(self, question: str) -> dict:
        result = rag_answer(question, self.collection, self.model)
        self._history.append({
            'q': question,
            'a': result['answer'],
            'sources': result['sources'],
        })
        return result

    def summary(self) -> str:
        return f'Questions asked: {len(self._history)}'


## Knowledge Base

In [ ]:
PYTHON_NOTES = 'Python is a high-level, general-purpose programming language known for its clean and readable syntax. It was created by Guido van Rossum and first released in 1991. Python supports multiple programming paradigms including procedural, object-oriented, and functional programming. It is widely used in data science, machine learning, web development, and automation. The Python Package Index (PyPI) hosts hundreds of thousands of third-party libraries. Python uses indentation to define code blocks, which enforces readable code structure. Common data structures in Python include lists, tuples, dictionaries, and sets.'

AI_HISTORY = 'The field of artificial intelligence began in the 1950s. Alan Turing proposed the Turing Test in 1950 as a measure of machine intelligence. The Dartmouth Conference in 1956 is widely regarded as the birthplace of AI as a formal discipline. Early AI research focused on symbolic reasoning and rule-based systems. The development of neural networks began in the 1950s with the perceptron, invented by Frank Rosenblatt. AI went through periods of reduced funding known as AI winters in the 1970s and 1980s. The deep learning revolution began around 2012 when convolutional neural networks dramatically improved image recognition accuracy.'

LLM_CONCEPTS = 'Large language models are neural networks trained on vast amounts of text data. They learn statistical patterns in language and can generate coherent text. The transformer architecture, introduced in 2017 in the paper Attention Is All You Need, is the foundation of modern LLMs. Transformers use self-attention mechanisms to model relationships between tokens in a sequence. GPT models are autoregressive, generating one token at a time based on the preceding context. Retrieval-augmented generation (RAG) combines LLMs with external knowledge retrieval to reduce hallucination. Context window refers to the maximum number of tokens a model can process at once.'

DOCS = {
    'python_notes.txt':  PYTHON_NOTES,
    'ai_history.txt':    AI_HISTORY,
    'llm_concepts.txt':  LLM_CONCEPTS,
}

bot = SecondBrainChatbot(DOCS)
print("Knowledge base indexed.")


## Scripted Q&A Session

In [ ]:
QUESTIONS = ['What programming language is known for clean syntax and was created by Guido van Rossum?', 'When did the field of artificial intelligence begin, and who proposed the Turing Test?', 'What architecture is the foundation of modern large language models?']

for i, q in enumerate(QUESTIONS, 1):
    print(f'--- Q{i} ---')
    print(f'Q: {q}')
    result = bot.ask(q)
    print(f"A: {result['answer'][:300]}")
    print(f"Sources: {result['sources']}")
    print()


## Mini Eval Run

In [ ]:
from dataclasses import dataclass, field


@dataclass
class TestCase:
    question: str
    expected_keywords: list[str] = field(default_factory=list)


def contains_any(response: str, keywords: list[str]) -> bool:
    resp_lower = response.lower()
    return any(kw.lower() in resp_lower for kw in keywords)


EVAL_CASES = [
    TestCase(
        'What is Python used for?',
        expected_keywords=['data science', 'machine learning', 'automation', 'web'],
    ),
    TestCase(
        'What is the Turing Test?',
        expected_keywords=['intelligence', 'machine', 'turing'],
    ),
    TestCase(
        'What does RAG stand for and what does it do?',
        expected_keywords=['retrieval', 'generation', 'hallucination', 'knowledge'],
    ),
]


In [ ]:
eval_results = []
for tc in EVAL_CASES:
    answer = bot.ask(tc.question)['answer']
    matched = [kw for kw in tc.expected_keywords
               if kw.lower() in answer.lower()]
    ok = bool(matched) if tc.expected_keywords else True
    icon = '\u2705' if ok else '\u274c'
    print(f'{icon} {tc.question[:60]}')
    if matched:
        print(f'   Keywords: {matched}')
    eval_results.append(ok)

pass_rate = sum(eval_results) / len(eval_results) if eval_results else 0.0
print(f'\nEval pass rate: {pass_rate*100:.1f}%')
print(bot.summary())
